# PubMedQA Data Exploration and Chunking Strategy

This notebook documents the dataset investigation used to inform the biomedical RAG ingestion and chunking design.

The goal is to make engineering decisions from the observed data rather than choosing parameters arbitrarily.


## 1. Load the dataset

The current project uses a 500-record subset of the labelled PubMedQA training data.


In [1]:
import json
import statistics
from pathlib import Path

DATA_PATH = Path("../data/pubmedqa.jsonl")

with DATA_PATH.open() as f:
    records = [json.loads(line) for line in f]

print(f"Records: {len(records)}")


Records: 500


## 2. Inspect the record structure


In [2]:
record = records[0]

print("Record fields:", list(record.keys()))
print("Context type:", type(record["context"]))
print("Context fields:", list(record["context"].keys()))
print("Number of context fragments:", len(record["context"]["contexts"]))


Record fields: ['pubid', 'question', 'context', 'long_answer', 'final_decision']
Context type: <class 'dict'>
Context fields: ['contexts', 'labels', 'meshes', 'reasoning_required_pred', 'reasoning_free_pred']
Number of context fragments: 2


## 3. Inspect individual context fragments

The dataset stores each research context as multiple short text fragments. We measure their lengths to determine whether those fragments are suitable as independent RAG documents.


In [3]:
context_lengths = [
    len(context.split())
    for record in records
    for context in record["context"]["contexts"]
]

print(f"Context fragments: {len(context_lengths)}")
print(f"Minimum words: {min(context_lengths)}")
print(f"Median words: {statistics.median(context_lengths):.1f}")
print(f"Mean words: {statistics.mean(context_lengths):.1f}")
print(f"Maximum words: {max(context_lengths)}")


Context fragments: 1706
Minimum words: 1
Median words: 51.0
Mean words: 58.2
Maximum words: 219


## 4. Finding: individual fragments are too short

The individual context fragments are short. Applying a conventional fixed-size chunker directly to these fragments would produce little or no additional chunking.

Instead, we investigate combining all context fragments belonging to the same PubMedQA record into one source document. This preserves the complete context associated with each question and gives us a meaningful unit on which to evaluate chunking.


In [4]:
document_lengths = [
    len(" ".join(record["context"]["contexts"]).split())
    for record in records
]

print(f"Documents: {len(document_lengths)}")
print(f"Minimum words: {min(document_lengths)}")
print(f"Median words: {statistics.median(document_lengths):.1f}")
print(f"Mean words: {statistics.mean(document_lengths):.1f}")
print(f"Maximum words: {max(document_lengths)}")


Documents: 500
Minimum words: 49
Median words: 200.0
Mean words: 198.6
Maximum words: 398


## 5. Compare candidate chunk sizes

We compare several reasonable chunk sizes instead of selecting one arbitrarily. Overlap is kept at 20% for this initial comparison.


In [5]:
def count_chunks(text, chunk_size, overlap):
    words = text.split()
    if not words:
        return 0

    step = chunk_size - overlap
    return max(1, (len(words) - overlap + step - 1) // step)


combined_documents = [
    " ".join(record["context"]["contexts"])
    for record in records
]

for chunk_size in [100, 150, 200]:
    overlap = int(chunk_size * 0.2)
    chunk_counts = [
        count_chunks(text, chunk_size, overlap)
        for text in combined_documents
    ]

    print(
        f"Chunk size: {chunk_size:>3} | "
        f"Overlap: {overlap:>3} | "
        f"Total chunks: {sum(chunk_counts):>4} | "
        f"Average/document: {statistics.mean(chunk_counts):.2f}"
    )


Chunk size: 100 | Overlap:  20 | Total chunks: 1376 | Average/document: 2.75
Chunk size: 150 | Overlap:  30 | Total chunks:  947 | Average/document: 1.89
Chunk size: 200 | Overlap:  40 | Total chunks:  753 | Average/document: 1.51


## 6. Inspect the actual chunking behaviour

Before selecting a strategy, inspect representative documents at the candidate sizes. This prevents choosing a parameter solely from aggregate statistics.


In [6]:
def chunk_text(text, chunk_size=150, overlap=30):
    words = text.split()
    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk = words[start:start + chunk_size]
        if chunk:
            chunks.append(" ".join(chunk))

    return chunks


sample_text = combined_documents[0]

for size in [100, 150, 200]:
    overlap = int(size * 0.2)
    chunks = chunk_text(sample_text, size, overlap)

    print(f"\nChunk size {size}, overlap {overlap}: {len(chunks)} chunks")
    for i, chunk in enumerate(chunks, 1):
        print(f"  Chunk {i}: {len(chunk.split())} words")



Chunk size 100, overlap 20: 4 chunks
  Chunk 1: 100 words
  Chunk 2: 100 words
  Chunk 3: 91 words
  Chunk 4: 11 words

Chunk size 150, overlap 30: 3 chunks
  Chunk 1: 150 words
  Chunk 2: 131 words
  Chunk 3: 11 words

Chunk size 200, overlap 40: 2 chunks
  Chunk 1: 200 words
  Chunk 2: 91 words


## 7. Final chunking rationale

The exploration showed that the individual context fragments are too short to make direct fixed-size chunking meaningful. Across 1,706 fragments, the median length is approximately 51 words.

Combining all context fragments belonging to each PubMedQA record produces 500 source documents with the following observed lengths:

- Minimum: 49 words
- Median: 200 words
- Mean: 198.6 words
- Maximum: 398 words

We then compared fixed-size approaches. Smaller chunks produce substantially more fragments, but naive word-based splitting can create awkward boundaries and very small trailing chunks. For example, a 150-word target produced a final 11-word chunk in the representative document.

The initial implementation therefore uses the dataset's existing context passages as semantic boundaries rather than splitting blindly on individual words. Context passages are combined until approximately 150 words are reached, with the final context carried into the next chunk as overlap. Very small trailing chunks are merged back into the preceding chunk.

The 150-word target is an initial engineering choice, not a claim that it is universally optimal. Retrieval quality will be evaluated later, and the chunking strategy can be revised based on measured retrieval performance.

This notebook records the analysis and rationale. The reusable chunking implementation belongs in `src/chunking.py`.

### Candidate strategy comparison

The initial word-based comparison produced:

| Target | Overlap | Total chunks | Avg. chunks/document |
|---:|---:|---:|---:|
| 100 words | 20 | 1,376 | 2.75 |
| 150 words | 30 | 947 | 1.89 |
| 200 words | 40 | 753 | 1.51 |

The final implementation deliberately does not use these word-level boundaries directly. The comparison was useful for understanding fragmentation; semantic context boundaries are used in the implementation instead.